## RF提取


In [18]:
# count_class_samples.py
import numpy as np
from WFlib.tools import data_processor
import torch

# 配置
feature_type = 'TAM'
seq_len = 1800

train_file = './datasets/TemporalDrift/tam_train.npz'
day14_file = './datasets/TemporalDrift/tam_day14.npz'

selected_classes = [0, 1, 2, 3, 4]

# 加载数据
def load_labels(file_path):
    _, y = data_processor.load_data(file_path, feature_type, seq_len)
    # 确保是 numpy 数组
    if isinstance(y, torch.Tensor):
        y = y.cpu().numpy()
    return y

y_train = load_labels(train_file)
y_day14 = load_labels(day14_file)

# 统计每个类别数量
print("类别样本数量统计:")
for cls in selected_classes:
    n_train = np.sum(y_train == cls)
    n_day14 = np.sum(y_day14 == cls)
    print(f"类别 {cls}: train = {n_train}, day14 = {n_day14}")

类别样本数量统计:
类别 0: train = 194, day14 = 223
类别 1: train = 193, day14 = 226
类别 2: train = 184, day14 = 225
类别 3: train = 165, day14 = 230
类别 4: train = 193, day14 = 227


In [24]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from WFlib import models, data_processor

# -------------------------------
# 配置
# -------------------------------
device = 'cuda:5'  # 对应你执行的命令
dataset = 'TemporalDrift'
feature_type = 'TAM'
seq_len = 1800
num_tabs = 1
selected_classes = [0,1,2,3,4]  # 只看前5类

# 模型权重路径
model_file = f'./checkpoints/{dataset}/RF/max_f1.pth'

# 数据路径
data_path = f'./datasets/{dataset}'
train_npz = os.path.join(data_path, 'tam_train.npz')
day14_npz = os.path.join(data_path, 'tam_valid.npz')  # day14 对应 valid

# -------------------------------
# 加载数据
# -------------------------------
train_X, train_y = data_processor.load_data(train_npz, feature_type, seq_len, num_tabs)
day14_X, day14_y = data_processor.load_data(day14_npz, feature_type, seq_len, num_tabs)

# -------------------------------
# 只选取类别0~4
# -------------------------------
train_mask = np.isin(train_y, selected_classes)
day14_mask = np.isin(day14_y, selected_classes)

train_X = train_X[train_mask]
train_y = train_y[train_mask]

day14_X = day14_X[day14_mask]
day14_y = day14_y[day14_mask]

# -------------------------------
# 调整输入形状 [N, 1, 2, L] -> [N, 2, L]
# -------------------------------
if train_X.ndim == 4 and train_X.shape[1] == 1:
    train_X = train_X.squeeze(1)
if day14_X.ndim == 4 and day14_X.shape[1] == 1:
    day14_X = day14_X.squeeze(1)

# -------------------------------
# 构建特征提取模型
# -------------------------------
num_classes = len(selected_classes)
model = models.RF(num_classes)  # RF 模型
checkpoint = torch.load(model_file, map_location=device)
model.load_state_dict(checkpoint, strict=False)
model.to(device)
model.eval()

# FeatureExtractor，截取最后 flatten 前的 features
class FeatureExtractor(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.features = base_model.features  # RF 的 feature 提取部分

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return x

feature_model = FeatureExtractor(model).to(device)
feature_model.eval()

# -------------------------------
# 提取特征
# -------------------------------
def extract_features(model, X):
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        if X_tensor.dim() == 4 and X_tensor.size(1) == 1:
            X_tensor = X_tensor.squeeze(1)
        feats = model(X_tensor)
        return feats.cpu().numpy()

train_feats = extract_features(feature_model, train_X)
day14_feats = extract_features(feature_model, day14_X)

print("Train features shape:", train_feats.shape)
print("Day14 features shape:", day14_feats.shape)

# -------------------------------
# t-SNE 可视化
# -------------------------------
X_all = np.concatenate([train_feats, day14_feats], axis=0)
y_all = np.concatenate([train_y, day14_y], axis=0)
domain_labels = np.array([0]*len(train_feats) + [1]*len(day14_feats))  # 0=train, 1=day14

tsne = TSNE(n_components=2, random_state=1013, perplexity=30)
X_2d = tsne.fit_transform(X_all)

plt.figure(figsize=(8,6))
colors = ['r','g','b','c','m']
markers = ['o','s']

for cls in selected_classes:
    # train
    mask = (y_all==cls) & (domain_labels==0)
    plt.scatter(X_2d[mask,0], X_2d[mask,1], c=colors[cls], marker='o', label=f"train_cls{cls}", alpha=0.5)
    # day14
    mask = (y_all==cls) & (domain_labels==1)
    plt.scatter(X_2d[mask,0], X_2d[mask,1], c=colors[cls], marker='s', label=f"day14_cls{cls}", alpha=0.7)

plt.legend()
plt.title(f"TAM Feature t-SNE ({dataset})")
plt.xlabel("t-SNE dim 1")
plt.ylabel("t-SNE dim 2")
plt.show()

类别样本数量统计:
类别 0: train=194, day14=223
类别 1: train=193, day14=226
类别 2: train=184, day14=225
类别 3: train=165, day14=230
类别 4: train=193, day14=227
类别 5: train=196, day14=228
类别 6: train=193, day14=217
类别 7: train=198, day14=219
类别 8: train=195, day14=224
类别 9: train=176, day14=231


/tmp/ipykernel_51476/913059699.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_file, map_location=device)


RuntimeError: Expected 2D (unbatched) or 3D (batched) input to conv1d, but got input of size: [1887, 1, 2, 1800]